<a href="https://colab.research.google.com/github/keondreblankson/WEEK8/blob/main/WEEK8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 Assignment 6: Titanic Data Analysis
Keondre Blankson

In this assignment, I will explore the Titanic passenger dataset.
I want to see how survival rates differed by passenger class and sex,
and look at the ages of the passengers. I will load the data directly
from an online source, handle missing values, create new columns,
and make charts to explore the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

df = pd.read_csv(url)

print("Rows and columns:", df.shape)
display(df.head())

In [ ]:
df.info()

print("\nMissing values in each column:")
display(df.isna().sum())

print("\nSummary statistics:")
display(df.describe())

## Cleaning the Data

I will keep the columns needed for this analysis. Some passengers
have missing ages, so I will replace those values with the median
of the recorded ages. I will also keep a column showing which ages
were originally missing. The age chart will only use recorded ages
so the replacement values do not distort the chart.

I will not automatically delete matching rows because different
passengers could have the same values in these columns.

In [ ]:
columns = [
    "survived", "pclass", "sex", "age",
    "sibsp", "parch", "fare"
]

clean_df = df[columns].copy()

# Record which ages were missing before filling them.
clean_df["age_was_missing"] = clean_df["age"].isna()

median_age = clean_df["age"].median()
clean_df["age"] = clean_df["age"].fillna(median_age)

# Standardize the text values.
clean_df["sex"] = clean_df["sex"].str.strip().str.lower()

print("Ages replaced:", clean_df["age_was_missing"].sum())
print("Median age used:", median_age)

print("\nMissing values after cleaning:")
display(clean_df.isna().sum())

## Transforming the Data

I will create a family size column by adding the number of siblings,
spouses, parents, and children traveling with each passenger, plus
the passenger. I will also change the passenger class numbers into
labels that are easier to read. Then I will group the data to
calculate survival percentages.

In [ ]:
clean_df["family_size"] = (
    clean_df["sibsp"] + clean_df["parch"] + 1
)

class_labels = {
    1: "First class",
    2: "Second class",
    3: "Third class"
}

clean_df["passenger_class"] = clean_df["pclass"].map(class_labels)

# Survived is 1 for survivors and 0 for others.
# Its mean multiplied by 100 gives the survival percentage.
survival_by_sex = (
    clean_df.groupby("sex")["survived"].mean() * 100
)

survival_by_class = (
    clean_df.groupby("passenger_class")["survived"]
    .mean()
    .reindex(["First class", "Second class", "Third class"])
    * 100
)

print("Survival percentage by sex:")
display(survival_by_sex.round(2))

print("\nSurvival percentage by passenger class:")
display(survival_by_class.round(2))

print("\nAverage family size:")
print(round(clean_df["family_size"].mean(), 2))

survival_by_sex.plot(
    kind="bar",
    color=["steelblue", "darkorange"],
    figsize=(7, 5)
)

plt.title("Titanic Survival Percentage by Sex")
plt.xlabel("Sex")
plt.ylabel("Passengers Who Survived (%)")
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
survival_by_class.plot(
    kind="bar",
    color=["seagreen", "steelblue", "darkorange"],
    figsize=(7, 5)
)

plt.title("Titanic Survival Percentage by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Passengers Who Survived (%)")
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
recorded_ages = clean_df.loc[
    ~clean_df["age_was_missing"], "age"
]

plt.figure(figsize=(8, 5))
plt.hist(
    recorded_ages,
    bins=20,
    color="steelblue",
    edgecolor="black"
)

plt.title("Distribution of Recorded Passenger Ages")
plt.xlabel("Age in Years")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.show()

## Conclusion

From this analysis, I found that female passengers had a higher
survival percentage than male passengers. First-class passengers
also had a higher survival percentage than second- and third-class
passengers. The age chart showed that many passengers with recorded
ages were young adults.

Cleaning and grouping the data made these patterns easier to see.
However, some ages were missing, so the age chart does not represent
every passenger. These comparisons show relationships in this
dataset, but they do not prove that one factor alone caused a
passenger to survive.

## Data Source

Titanic dataset from the Seaborn data repository:
https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv